## Step 1: Install dependencies

In [ ]:
!pip install -q sentence-transformers datasets pinecone weaviate-client qdrant-client chromadb

import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import pinecone
import weaviate
from qdrant_client import QdrantClient
from qdrant_client.http import models
import chromadb
import uuid

## Step 2: Load and prepare the Quora Question Pairs dataset
Using a subset of 1000 questions for demo, with topic metadata

In [ ]:
dataset = load_dataset("squad", split="train[:1000]")
questions = [item['question'] for item in dataset]
# Create synthetic metadata for filtering (e.g., assign categories based on context length)
metadata = [{"id": str(uuid.uuid4()), "category": "long" if len(item['context'].split()) > 100 else "short"}
            for item in dataset]

## Step 3: Initialize the embedding model
Using all-MiniLM-L6-v2 for 384-dimensional embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(questions, batch_size=32, show_progress_bar=True)

## Step 4: ChromaDB - Lightweight Embeddable Vector Database

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="squad_questions")

In [ ]:
# Add vectors with metadata
collection.add(
    embeddings=embeddings.tolist(),
    documents=questions,
    metadatas=metadata,
    ids=[meta["id"] for meta in metadata]
)

In [ ]:
query = "Pakistan"
query_emb = model.encode([query])[0]

In [ ]:
# Query ChromaDB
chroma_results = collection.query(
    query_embeddings=query_emb.tolist(),
    n_results=3,
    where={"category": "short"}
)
print("\nChromaDB Results:")
for id, doc, dist in zip(chroma_results["ids"][0], chroma_results["documents"][0], chroma_results["distances"][0]):
    print(f"ID: {id}, Text: {doc}, Distance: {dist}")
